In [16]:
import torch
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

In [17]:
import torch
ground_coarse_token = []
ground_fine_token = []
for i in range(1, 6):
    checkpoint_path = f"/data/esmdiff_target/bpti_kinetic_cluster/bpti_ptm/bpti_{i}.ptm"
    checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))
    ground_coarse_token.append(checkpoint['small_codebook'])
    ground_fine_token.append(checkpoint['large_codebook'])

In [18]:
ground_coarse_token

[tensor([34, 27,  4, 19, 12,  2, 12, 31, 29, 29, 31, 13, 16, 19, 25, 19, 31, 13,
          3,  4,  3, 15, 13, 15, 31, 26, 19,  4, 14,  4, 15,  3, 15, 31, 15, 13,
         27, 14, 19, 16,  3,  4, 10,  2, 13,  3, 19, 29, 17, 19, 24,  2, 11, 19,
         10, 27,  6, 29, 10, 33]),
 tensor([34, 27,  4, 19, 26,  2, 12, 25, 15,  2,  3, 18, 29,  4, 13, 24, 31, 16,
         31, 25,  3, 27, 13, 15, 31, 26, 19,  4, 14, 29, 15, 31, 15,  3, 27,  3,
         18, 14, 27, 29,  3, 15, 10, 12,  3,  3, 19, 29, 17, 19, 24,  2, 12, 24,
         19,  5, 18,  3, 10, 33]),
 tensor([34, 27, 25, 19, 12, 12, 31, 29, 28,  4, 15, 18, 29, 29, 28, 18, 25, 25,
          6,  4,  3, 27, 13, 27, 31, 19, 19, 25, 14, 29, 15, 31, 15, 31, 15, 31,
         13, 14,  2, 13, 29, 10, 12, 16,  3,  3, 11, 29, 17, 19, 19,  2, 26, 24,
         11, 26,  6, 18, 25, 33]),
 tensor([34, 27, 11, 14, 12, 26, 18, 25, 29, 29,  3,  3, 16, 10, 25,  4, 27, 18,
         25,  4,  3, 15, 13, 15, 31, 12, 19,  4, 14,  4, 15, 31, 15, 31, 15,  3,
    

In [19]:
for j in range(ground_coarse_token[0].shape[0]):
    for i in range(5):
        print(f"{ground_coarse_token[i][j]:>4}", end=" ")
    print()

  34   34   34   34   34 
  27   27   27   27   27 
   4    4   25   11    4 
  19   19   19   14   19 
  12   26   12   12   12 
   2    2   12   26   12 
  12   12   31   18   12 
  31   25   29   25   31 
  29   15   28   29   29 
  29    2    4   29   15 
  31    3   15    3   25 
  13   18   18    3   13 
  16   29   29   16   16 
  19    4   29   10   19 
  25   13   28   25   25 
  19   24   18    4   29 
  31   31   25   27    6 
  13   16   25   18   24 
   3   31    6   25   31 
   4   25    4    4   27 
   3    3    3    3   13 
  15   27   27   15   27 
  13   13   13   13   13 
  15   15   27   15   15 
  31   31   31   31   31 
  26   26   19   12   26 
  19   19   19   19   19 
   4    4   25    4    4 
  14   14   14   14   14 
   4   29   29    4   29 
  15   15   15   15   15 
   3   31   31   31   31 
  15   15   15   15   15 
  31    3   31   31   31 
  15   27   15   15   15 
  13    3   31    3    3 
  27   18   13   28   27 
  14   14   14   14   14 
  19   27   

In [20]:
stacked_tensor = torch.stack(ground_coarse_token)
data_matrix = stacked_tensor.numpy()
df_view = pd.DataFrame(data_matrix.T, columns=[f"Sample_{i}" for i in range(5)])

print("=== 데이터 형태 (상위 5행) ===")
print(df_view.head())
print("-" * 50)

=== 데이터 형태 (상위 5행) ===
   Sample_0  Sample_1  Sample_2  Sample_3  Sample_4
0        34        34        34        34        34
1        27        27        27        27        27
2         4         4        25        11         4
3        19        19        19        14        19
4        12        26        12        12        12
--------------------------------------------------


In [21]:
unique_counts = df_view.nunique(axis=1)

# 4. 조건 필터링: 종류가 3개 이상인 행만 추출
diverse_rows = df_view[unique_counts >= 5].copy()

# 보기 좋게 'Unique_Count' 컬럼 추가
diverse_rows['Unique_Count'] = unique_counts[unique_counts >= 5]

print(f"=== 전체 60개 위치 중 다양성이 높은(5종류 이상) 위치: {len(diverse_rows)}곳 ===")
print(diverse_rows)

=== 전체 60개 위치 중 다양성이 높은(5종류 이상) 위치: 3곳 ===
    Sample_0  Sample_1  Sample_2  Sample_3  Sample_4  Unique_Count
15        19        24        18         4        29             5
17        13        16        25        18        24             5
41         4        15        10        31        28             5


In [22]:
raw_indices = diverse_rows.index
for i in raw_indices: 
    print(i, end="A+")

15A+17A+41A+

In [23]:
unique_counts = df_view.nunique(axis=1)

# 4. 조건 필터링: 종류가 3개 이상인 행만 추출
diverse_rows = df_view[unique_counts >= 3].copy()

# 보기 좋게 'Unique_Count' 컬럼 추가
diverse_rows['Unique_Count'] = unique_counts[unique_counts >= 3]

print(f"=== 전체 60개 위치 중 다양성이 높은(3종류 이상) 위치: {len(diverse_rows)}곳 ===")
print(diverse_rows)

=== 전체 60개 위치 중 다양성이 높은(3종류 이상) 위치: 29곳 ===
    Sample_0  Sample_1  Sample_2  Sample_3  Sample_4  Unique_Count
2          4         4        25        11         4             3
5          2         2        12        26        12             3
6         12        12        31        18        12             3
7         31        25        29        25        31             3
8         29        15        28        29        29             3
9         29         2         4        29        15             4
10        31         3        15         3        25             4
11        13        18        18         3        13             3
13        19         4        29        10        19             4
14        25        13        28        25        25             3
15        19        24        18         4        29             5
16        31        31        25        27         6             4
17        13        16        25        18        24             5
18         3      

In [24]:
raw_indices = diverse_rows.index
for i in raw_indices:
    print(i, end="A+")

2A+5A+6A+7A+8A+9A+10A+11A+13A+14A+15A+16A+17A+18A+19A+25A+35A+36A+38A+39A+40A+41A+42A+43A+52A+54A+55A+56A+57A+

In [25]:
raw_indices = diverse_rows.index
for i in raw_indices:
    print(i, end=",")

2,5,6,7,8,9,10,11,13,14,15,16,17,18,19,25,35,36,38,39,40,41,42,43,52,54,55,56,57,

In [26]:
unique_counts = df_view.nunique(axis=1)

# 4. 조건 필터링: 종류가 3개 이상인 행만 추출
diverse_rows = df_view[unique_counts >= 4].copy()

# 보기 좋게 'Unique_Count' 컬럼 추가
diverse_rows['Unique_Count'] = unique_counts[unique_counts >= 4]

print(f"=== 전체 60개 위치 중 다양성이 높은(4종류 이상) 위치: {len(diverse_rows)}곳 ===")
print(diverse_rows)

=== 전체 60개 위치 중 다양성이 높은(4종류 이상) 위치: 14곳 ===
    Sample_0  Sample_1  Sample_2  Sample_3  Sample_4  Unique_Count
9         29         2         4        29        15             4
10        31         3        15         3        25             4
13        19         4        29        10        19             4
15        19        24        18         4        29             5
16        31        31        25        27         6             4
17        13        16        25        18        24             5
18         3        31         6        25        31             4
36        27        18        13        28        27             4
39        16        29        13         4        16             4
40         3         3        29        31        13             4
41         4        15        10        31        28             5
54        10        19        11        17        10             4
55        27         5        26        17        17             4
57        29      

In [27]:
raw_indices = diverse_rows.index
for i in raw_indices:
    print(i, end="A+")

9A+10A+13A+15A+16A+17A+18A+36A+39A+40A+41A+54A+55A+57A+

In [28]:
raw_indices = diverse_rows.index

# 2. PDB 매핑을 위한 보정 (+1) 및 리스트 변환
# PDB 파일의 residue 번호가 1번부터 시작한다고 가정할 때:
target_indices = (raw_indices + 1).tolist()

print(f"다양성이 높은 위치(0-based): {raw_indices.tolist()}")
print(f"PDB 시각화용 타겟(1-based): {target_indices}")

다양성이 높은 위치(0-based): [9, 10, 13, 15, 16, 17, 18, 36, 39, 40, 41, 54, 55, 57]
PDB 시각화용 타겟(1-based): [10, 11, 14, 16, 17, 18, 19, 37, 40, 41, 42, 55, 56, 58]


In [29]:
for i in raw_indices:
    print(i, end="A+")

9A+10A+13A+15A+16A+17A+18A+36A+39A+40A+41A+54A+55A+57A+

In [30]:
import py3Dmol
import os

def visualize_pretty_cartoon(pdb_file1, pdb_file2, highlight_residues):
    # 인덱스 안전 변환
    if highlight_residues:
        highlight_residues = [int(x) for x in highlight_residues]
    else:
        highlight_residues = []

    # 파일 읽기
    with open(pdb_file1, 'r') as f: data1 = f.read()
    with open(pdb_file2, 'r') as f: data2 = f.read()

    print(f"=== [구조 1] {os.path.basename(pdb_file1)} ===")
    view1 = py3Dmol.view(width=700, height=500)
    view1.addModel(data1, 'pdb')
    
    # -----------------------------------------------------------
    # [스타일 1] 베이스 스타일 (전체 구조)
    # - arrows: true (Beta sheet 화살표)
    # - style: 'oval' (단면을 둥글게 -> Helix가 두툼해 보임)
    # - color: 'spectrum' (N->C 무지개색) 혹은 단색
    # -----------------------------------------------------------
    view1.setStyle({
        'cartoon': {
            'color': 'white',      # 기본 흰색/회색
            'arrows': True,        # 화살표 켜기 (필수)
            'style': 'oval',       # 단면을 타원형으로 (입체감)
            'thickness': 0.5,      # 기본 두께
            'opacity': 0.7
        }
    })
    
    # -----------------------------------------------------------
    # [스타일 2] 강조 스타일 (Target Residues)
    # - thickness: 1.0 (두껍게)
    # - color: 'red' (빨간색)
    # -----------------------------------------------------------
    if highlight_residues:
        view1.addStyle(
            {'resi': highlight_residues}, 
            {'cartoon': {
                'color': 'red', 
                'arrows': True,    # 여기도 화살표 유지
                'style': 'oval',
                'thickness': 1.0,  # 강조 부분은 더 두껍게!
                'opacity': 1.0
            }}
        )
        
    view1.zoomTo()
    view1.show()

    print(f"\n=== [구조 2] {os.path.basename(pdb_file2)} ===")
    view2 = py3Dmol.view(width=700, height=500)
    view2.addModel(data2, 'pdb')
    
    # 구조 2에도 동일한 스타일 적용 (비교를 위해 파란색 계열로)
    view2.setStyle({
        'cartoon': {
            'color': '#6495ED',    # 옥수수 파란색 (CornflowerBlue)
            'arrows': True,
            'style': 'oval',
            'thickness': 0.5,
            'opacity': 0.7
        }
    })
    
    if highlight_residues:
        view2.addStyle(
            {'resi': highlight_residues}, 
            {'cartoon': {
                'color': 'red', 
                'arrows': True,
                'style': 'oval',
                'thickness': 1.0,
                'opacity': 1.0
            }}
        )

    view2.zoomTo()
    view2.show()

# --- 실행 ---
# target_indices가 정의되어 있어야 합니다.
visualize_pretty_cartoon(
    '/data/esmdiff_target/bpti_kinetic_cluster/bpti_5/bpti_1.pdb', 
    '/data/esmdiff_target/bpti_kinetic_cluster/bpti_5/bpti_3.pdb', 
    target_indices
)

=== [구조 1] bpti_1.pdb ===


3Dmol.js failed to load for some reason. Please check your browser console for error messages.


=== [구조 2] bpti_3.pdb ===


3Dmol.js failed to load for some reason. Please check your browser console for error messages.